# Milestone 3 walkthrough: Experiment 5 (Pipelines A and B)

This notebook runs nothing heavy by default. It **reads the stored outputs** of every stage in
`experiments/experiment_5/` and shows them, so you can follow the whole experiment without refitting.
Set `RUN_REPRODUCTION = True` to rerun the pipelines from the raw data and compare them with the release
(about 25 minutes; see `REPRODUCE.md` at the repository root).

- **Pipeline A** (`experiments/experiment_5/stages/`): all states, prediction at checkout, stages s01–s08.
- **Pipeline B** (`experiments/experiment_5/pipeline_b/`): seven states, seller and route history,
  classification at carrier handover, stages B2–B12.
- **Joint test T1:** the held-out test was read once for both pipelines and is never rerun.

Every number the report quotes comes from the register `pipeline_b/evidence/registry_B.yaml`, shown in
section 5.

In [ ]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

REPO = Path.cwd().resolve()
while not (REPO / "experiments" / "experiment_5").exists():
    REPO = REPO.parent
E5 = REPO / "experiments" / "experiment_5"
B = E5 / "pipeline_b"
PY = E5 / ".venv" / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
RUN_REPRODUCTION = False   # True reruns A s01-s08 and B B2-B12 from data/raw (about 25 minutes)
pd.set_option("display.max_columns", 30)
print("repository:", REPO)

def show_csv(path, n=20):
    df = pd.read_csv(path)
    display(Markdown(f"`{path.relative_to(REPO)}`")); display(df.head(n))

def show_fig(path, width=820):
    display(Markdown(f"`{path.relative_to(REPO)}`")); display(Image(filename=str(path), width=width))

## 1. Optional: rerun and compare

The rerun writes new outputs in place, then `compare` checks every file against the release tag
`e5-release-2` and writes `reproduction/reproduction_check.json`. Run it on a clean checkout of the tag.

In [ ]:
if RUN_REPRODUCTION:
    subprocess.run([str(PY), str(E5 / "reproduce.py"), "run"], cwd=REPO, check=True)
    subprocess.run([str(PY), str(E5 / "reproduce.py"), "compare", "--machine", "other"], cwd=REPO)
check = E5 / "reproduction" / "reproduction_check.json"
if check.exists():
    r = json.loads(check.read_text(encoding="utf-8"))
    print({k: r[k] for k in ("release_tag", "machine", "rule", "verdict", "summary")})
else:
    print("No reproduction check in this checkout yet.")

## 2. Pipeline A: status of every stage

Each stage writes its outputs with a manifest (run ID, code hash, inputs). `ACCEPTED` means the team
approved the stage; s08's gate fed the joint test T1, which replaced A's s09 and s10 (decision G-79).

In [ ]:
print(subprocess.run([str(PY), str(E5 / "run_pipeline.py"), "status"], cwd=REPO,
                     capture_output=True, text=True).stdout)

## 3. Pipeline A: data, partition and model selection

In [ ]:
show_csv(E5 / "stages/s02_data/common/outputs/exclusion_counts.csv")       # who is excluded, by rule
for task in ("regression", "classification"):
    show_csv(E5 / f"stages/s05_partition/{task}/outputs/fold_summary.csv")   # chronological, customer-purged folds
    show_csv(E5 / f"stages/s08_selection/{task}/outputs/ladder.csv")        # simplest-first ladder, one-SE rule

## 4. Pipeline B and the joint test

B narrows the scope to seven states and adds seller and route history (CR-054). Both pipelines were then
scored once on the same held-out test (T1).

In [ ]:
for task in ("regression", "classification"):
    show_csv(B / f"stages/b02/{task}/outputs/exclusion_counts_B.csv")
    show_csv(B / f"stages/t1_joint_test/{task}/outputs/metrics_test.csv")
    show_csv(B / f"stages/b09/{task}/outputs/train_val_test.csv")
    show_csv(B / f"stages/b09/{task}/outputs/paired_test_differences.csv")

In [ ]:
fig = B / "evidence" / "build" / "figures"
for name in ("F.reg.partition", "F.reg.selection", "F.reg.validation_and_test", "F.clf.validation_and_test",
             "F.clf.test_curves", "F.clf.threshold_options", "F.reg.residuals", "F.reg.by_state",
             "F.reg.importance", "F.clf.importance", "F.reg.drift"):
    show_fig(fig / f"{name}.png")

## 5. Checks added for the Milestone 2 feedback (B12)

Feature ranking repeated inside each training fold, and the promise buffer chosen on validation and read
on the test.

In [ ]:
show_csv(B / "stages/b12/regression/outputs/buffer_choice.csv")
show_csv(B / "stages/b12/regression/outputs/ranking_stability_by_fold.csv")
for name in ("F.reg.buffer_curve", "F.reg.ranking_stability", "F.clf.ranking_stability"):
    show_fig(fig / f"{name}.png")

## 6. The number register used by the report and slides

In [ ]:
nums = json.loads((B / "evidence/build/numbers_B.json").read_text(encoding="utf-8"))
reg = pd.DataFrame([{"id": k, "text": v.get("text"), "evidence": v.get("evidence"), "source": v.get("source")}
                    for k, v in nums.items() if v.get("kind") == "number"])
display(reg)

## 7. Deployment inputs

The app reads only stored files: the frozen bundles from T1 and B10's serving configuration. B scores
orders inside its seven states and A's frozen model the rest (B-G-06).

In [ ]:
for task in ("regression", "classification"):
    print(task, json.dumps(json.loads((B / f"stages/b10/{task}/outputs/serving_config.json").read_text(encoding="utf-8")), indent=1)[:1500])